# 👥 Lab W6-1 — RFM Segmentation ด้วย K-Means

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 6 — Data Mining II**

Lab นี้ใช้คู่กับสื่อจำลอง **Segment Studio** (`/sims/segment-studio`)
ค่า silhouette และการเลือก k ในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง
(ดูหมายเหตุเรื่อง inertia ท้ายเล่ม)

## สิ่งที่จะได้เรียนรู้
1. อธิบายด้วยตัวเลขว่าเหตุใด **ต้อง normalize ก่อน K-Means เสมอ**
2. เลือกค่า k ด้วย **Elbow · Silhouette · และเหตุผลทางธุรกิจ** ประกอบกัน
3. แปลผลการแบ่งกลุ่มให้กลายเป็น **ข้อเสนอเชิงธุรกิจที่มีตัวเลขกำกับ**
4. รู้ว่า **silhouette เทียบข้ามการปรับสเกลไม่ได้**

## ข้อมูล
`customers_rfm.csv` — ลูกค้า 1,200 รายพร้อมค่า Recency · Frequency · Monetary

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week06/customers_rfm.csv")
df = pd.read_csv(URL).sort_values("customer_id").reset_index(drop=True)

F = ["recency_days", "frequency", "monetary"]

print(f"จำนวนลูกค้า : {len(df):,}")
print(df[F].describe().loc[["min", "max", "mean", "std"]].to_string())
df.head(5)

> **สังเกตช่วงของแต่ละตัวแปร**
> `recency` 3–419 · `frequency` 1–48 · `monetary` 481–209,472
>
> ระยะทางแบบยุคลิดบวกกำลังสองของทุกแกนเข้าด้วยกัน — แกนที่มีช่วงกว้างกว่าจะครองผลลัพธ์ทั้งหมด

### 🧑‍💻 งานที่ 1
คำนวณว่าช่วง (max − min) ของ `monetary` กว้างกว่าช่วงของ `frequency` และ `recency` กี่เท่า
แล้วคำนวณว่าถ้าลูกค้าสองรายต่างกัน

* รายแรก: `frequency` ต่างกัน 47 ครั้ง (มากที่สุดเท่าที่เป็นไปได้) แต่ `monetary` เท่ากัน
* รายที่สอง: `monetary` ต่างกันเพียง 1,000 บาท แต่ตัวอื่นเท่ากัน

คู่ใดจะมีระยะทางแบบยุคลิดมากกว่ากัน และมากกว่ากี่เท่า

In [ ]:
rng_stats = df[F].max() - df[F].min()
print("ช่วงของแต่ละตัวแปร")
print(rng_stats.to_string())
print(f"\nmonetary กว้างกว่า frequency : {rng_stats.monetary/rng_stats.frequency:,.0f} เท่า")
print(f"monetary กว้างกว่า recency   : {rng_stats.monetary/rng_stats.recency_days:,.0f} เท่า")

d_freq = np.sqrt(47 ** 2)
d_money = np.sqrt(1000 ** 2)
print(f"\nระยะทางเมื่อ frequency ต่างกันสูงสุด 47 ครั้ง : {d_freq:,.2f}")
print(f"ระยะทางเมื่อ monetary ต่างกันเพียง 1,000 บาท : {d_money:,.2f}")
print(f"→ ยอดเงินที่ต่างกันแค่ 1,000 บาท มีน้ำหนักมากกว่า "
      f"ความต่างของพฤติกรรมสูงสุด {d_money/d_freq:.1f} เท่า")
print("""
K-Means จึงไม่ได้ 'เลือก' ที่จะสนใจยอดเงิน — มันไม่มีทางเลือกอื่นเลย
เพราะสูตรระยะทางบอกมันว่าแกนนั้นสำคัญกว่าแกนอื่นหลายพันเท่า
""")

## ส่วนที่ 2 — แบ่งกลุ่มสองแบบแล้วเทียบกัน

### 🧑‍💻 งานที่ 2
รัน K-Means ที่ k = 4 สองครั้ง — ครั้งแรกกับข้อมูลดิบ ครั้งที่สองกับข้อมูลที่ปรับสเกลแล้ว
แล้วสร้างตารางโปรไฟล์ (ค่าเฉลี่ย R · F · M และจำนวนสมาชิก) ของทั้งสองแบบ

ใช้ `random_state=42` และ `n_init=10` เพื่อให้ผลทำซ้ำได้

*เฉลยที่ถูกต้อง: แบบปรับสเกลจะได้กลุ่มขนาด 442 / 388 / 242 / 128
โดยมีกลุ่มที่ F ≈ 1.5 แต่ M ≈ 141,082 ซึ่งเป็นกลุ่มที่หายไปทั้งหมดถ้าไม่ปรับสเกล*

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

raw = df[F].astype(float).values
scaled = StandardScaler().fit_transform(raw)


def profile(X, k=4, label=""):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    p = df.assign(cl=km.labels_).groupby("cl")[F].mean()
    p["จำนวน"] = np.bincount(km.labels_, minlength=k)
    print(f"\n--- {label} (k={k}) ---")
    print(p.sort_values("จำนวน", ascending=False).to_string())
    return km


km_raw = profile(raw, 4, "ข้อมูลดิบ ไม่ปรับสเกล")
km_scaled = profile(scaled, 4, "ปรับสเกลแล้ว")

print("""
สิ่งที่ต้องสังเกต
-----------------
แบบไม่ปรับสเกล : ทั้ง 4 กลุ่มต่างกันแทบเฉพาะที่ยอดเงิน ส่วน R และ F ปะปนกัน
                  ผลที่ได้จึงเทียบเท่ากับ pd.qcut(monetary, 4) ซึ่งไม่ต้องใช้ K-Means เลย

แบบปรับสเกล    : ได้กลุ่มพฤติกรรมที่แยกกันจริง 4 กลุ่ม โดยเฉพาะกลุ่มที่
                  ซื้อครั้งเดียว (F ≈ 1.5) แต่ยอดสูงมาก (M ≈ 141,000)
                  ซึ่งมียอดเงินพอ ๆ กับกลุ่มแชมเปี้ยน แต่พฤติกรรมตรงข้ามกันสิ้นเชิง
                  ถ้าไม่ปรับสเกล สองกลุ่มนี้จะถูกยุบรวมกันทันที
""")

## ส่วนที่ 3 — เลือก k อย่างไร

### 🧑‍💻 งานที่ 3
คำนวณ inertia และ silhouette ของ k = 2 ถึง 8 บนข้อมูลที่ปรับสเกลแล้ว
แล้วตอบว่า

1. inertia ใช้เลือก k ตรง ๆ ได้หรือไม่ เพราะเหตุใด
2. silhouette ชี้ไปที่ k เท่าไร

*เฉลยที่ถูกต้อง: silhouette สูงสุดที่ k = 4 ได้ 0.5847*

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(scaled)
    rows.append({"k": k, "inertia": km.inertia_,
                 "silhouette": silhouette_score(scaled, km.labels_)})

curve = pd.DataFrame(rows).set_index("k")
curve["inertia ลดลง %"] = -curve.inertia.pct_change() * 100
print(curve.to_string())

best_k = curve.silhouette.idxmax()
print(f"\nsilhouette สูงสุดที่ k = {best_k} ({curve.silhouette.max():.4f})")

print("""
คำตอบข้อ 1
----------
inertia ลดลงเสมอเมื่อ k เพิ่ม และจะเป็นศูนย์พอดีเมื่อ k เท่ากับจำนวนลูกค้า
(ทุกคนเป็นกลุ่มของตัวเอง) จึงใช้เลือก k ตรง ๆ ไม่ได้ — ถ้าเลือกจาก 'inertia ต่ำสุด'
จะได้ k = 1,200 เสมอ ซึ่งไร้ความหมาย

สิ่งที่ต้องมองคือ 'ข้อศอก' — จุดที่การลดลงเริ่มชะลอลงชัดเจน
แต่ข้อศอกเป็นการตัดสินด้วยสายตา คนละคนอ่านได้ไม่เหมือนกัน
""")

## ส่วนที่ 4 — กับดักของ Silhouette

นี่คือส่วนที่นักศึกษาที่เชื่อตัววัดมากที่สุดมักติดกับ

### 🧑‍💻 งานที่ 4
คำนวณ silhouette ที่ k = 4 ของ **ข้อมูลดิบ** แล้วเทียบกับของ **ข้อมูลที่ปรับสเกลแล้ว**

ถ้าเลือกโมเดลด้วย silhouette เพียงอย่างเดียว จะได้โมเดลใดมาใช้งาน
และเหตุใดข้อสรุปนั้นจึงผิด

*เฉลยที่ถูกต้อง: ข้อมูลดิบได้ 0.6392 ซึ่ง **สูงกว่า** ข้อมูลที่ปรับสเกลแล้วที่ได้ 0.5847*

In [ ]:
sil_raw = silhouette_score(raw, km_raw.labels_)
sil_scaled = silhouette_score(scaled, km_scaled.labels_)

print(f"silhouette ของข้อมูลดิบ (k=4)        : {sil_raw:.4f}")
print(f"silhouette ของข้อมูลที่ปรับสเกล (k=4) : {sil_scaled:.4f}")
print(f"ต่างกัน                              : {sil_raw - sil_scaled:+.4f}")

print("""
เหตุใดจึงผิด
------------
silhouette คำนวณจากระยะทางใน 'หน่วยของข้อมูลที่ป้อนเข้าไป'
ข้อมูลดิบมีมิติเดียวที่ครองทุกอย่าง (monetary) การแบ่งช่วงบนแกนเดียว
ย่อมได้กลุ่มที่ 'กระชับ' ในสายตาของสูตรระยะทางเสมอ

ตัวเลขจึงสูงกว่าโดยธรรมชาติ ทั้งที่ผลลัพธ์ไร้ประโยชน์ทางธุรกิจ

กฎที่ต้องจำ
  ตัววัดภายในทุกตัว (silhouette · Davies-Bouldin · Calinski-Harabasz)
  ใช้เทียบได้เฉพาะ 'ค่า k ต่าง ๆ ภายใต้การเตรียมข้อมูลแบบเดียวกัน' เท่านั้น

  การเลือกวิธีเตรียมข้อมูลต้องตัดสินด้วยเหตุผลเชิงโดเมน ไม่ใช่ด้วยคะแนน
  และการที่คะแนนสูงกว่าไม่ได้แปลว่าคำตอบดีกว่า
""")

## ส่วนที่ 5 — จากกลุ่มสู่แคมเปญ

### 🧑‍💻 งานที่ 5
ตั้งชื่อกลุ่มทั้ง 4 ของโมเดลที่ปรับสเกลแล้วเป็นภาษาธุรกิจ
จากนั้นเสนอ **การกระทำที่ต่างกัน** สำหรับแต่ละกลุ่ม พร้อมประมาณการมูลค่า

สมมติฐานที่ให้ใช้: ต้นทุนติดต่อ 150 บาท/ราย · กำไรขั้นต้น 22% ของยอดซื้อ
และแคมเปญทำให้ลูกค้าในกลุ่มนั้นซื้อเพิ่มตามอัตราที่คุณประมาณเอง (ต้องระบุเหตุผล)

In [ ]:
CONTACT_COST = 150
MARGIN = 0.22

seg = df.assign(cl=km_scaled.labels_).groupby("cl").agg(
    n=("customer_id", "size"),
    R=("recency_days", "mean"),
    Fq=("frequency", "mean"),
    M=("monetary", "mean"),
)


def name_of(r):
    if r.Fq >= 20 and r.R < 90:
        return "🏆 แชมเปี้ยน"
    if r.Fq <= 3 and r.M > 60000:
        return "💎 ซื้อครั้งใหญ่ครั้งเดียว"
    if r.R > 150:
        return "😴 หลับใหล"
    return "🛒 ลูกค้าประจำ"


seg["กลุ่ม"] = seg.apply(name_of, axis=1)

# อัตราการตอบสนองที่ประมาณจากลักษณะของกลุ่ม — ต้องอธิบายเหตุผลได้
UPLIFT = {
    "🏆 แชมเปี้ยน": 0.04,              # ซื้ออยู่แล้ว เพิ่มได้ไม่มาก
    "🛒 ลูกค้าประจำ": 0.12,            # มีพื้นที่ให้เติบโตมากที่สุด
    "💎 ซื้อครั้งใหญ่ครั้งเดียว": 0.18,  # ยอดต่อครั้งสูง ถ้ากลับมาได้คุ้มมาก
    "😴 หลับใหล": 0.03,                # ปลุกยาก และยอดต่อรายต่ำ
}
seg["uplift"] = seg["กลุ่ม"].map(UPLIFT)
seg["ต้นทุนติดต่อ"] = seg.n * CONTACT_COST
seg["กำไรคาดหวัง"] = seg.n * seg.M * seg.uplift * MARGIN
seg["กำไรสุทธิ"] = seg["กำไรคาดหวัง"] - seg["ต้นทุนติดต่อ"]
seg["ROI %"] = seg["กำไรสุทธิ"] / seg["ต้นทุนติดต่อ"] * 100

out = seg.set_index("กลุ่ม")[["n", "R", "Fq", "M", "uplift",
                              "ต้นทุนติดต่อ", "กำไรสุทธิ", "ROI %"]]
print(out.sort_values("กำไรสุทธิ", ascending=False).to_string())
print(f"\nกำไรสุทธิรวมถ้าทำทุกกลุ่ม : {seg['กำไรสุทธิ'].sum():,.0f} บาท")

worst = seg.loc[seg["ROI %"].idxmin()]
print(f"\nกลุ่มที่ ROI ต่ำที่สุดคือ {worst['กลุ่ม']} ({worst['ROI %']:.0f}%)")
print("→ ถ้างบจำกัด ให้ตัดกลุ่มนี้ออกก่อน ไม่ใช่ตัดกลุ่มที่มีสมาชิกน้อยที่สุด")

## ส่วนที่ 6 — ทางเลือกอื่นและข้อจำกัด

### 🧑‍💻 งานที่ 6 (เขียนเป็นข้อความ)

1. ถ้าทีมการตลาดบอกว่าบริหารแคมเปญพร้อมกันได้ไม่เกิน 3 แบบ
   คุณจะเลือก k = 3 หรือยืนยัน k = 4 ตาม silhouette — ตอบพร้อมเหตุผล
2. ยกข้อจำกัดของ K-Means อย่างน้อย 3 ข้อที่ปรากฏในโจทย์นี้
3. Hierarchical clustering และ DBSCAN จะช่วยแก้ข้อใดได้บ้าง
4. ลูกค้าจะเปลี่ยนกลุ่มไปตามเวลา ควรรันการแบ่งกลุ่มใหม่บ่อยแค่ไหน
   และจะรู้ได้อย่างไรว่าถึงเวลาแล้ว

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

# เทียบกับอัลกอริทึมอื่นภายใต้ข้อมูลเดียวกัน
hier = fcluster(linkage(scaled, method="ward"), 4, criterion="maxclust")
db = DBSCAN(eps=0.55, min_samples=12).fit(scaled)

print(f"K-Means      : 4 กลุ่ม")
print(f"Hierarchical : {len(set(hier))} กลุ่ม  "
      f"· ตรงกับ K-Means (ARI) = {adjusted_rand_score(km_scaled.labels_, hier):.4f}")
print(f"DBSCAN       : {len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)} กลุ่ม "
      f"+ จุดที่ถือเป็น noise {(db.labels_ == -1).sum():,} จุด")

print("""
คำตอบข้อ 1
----------
เลือก k = 3 แล้วรวมสองกลุ่มที่ใกล้กันที่สุดเข้าด้วยกัน
เพราะกลุ่มที่บริหารไม่ได้ก็คือกลุ่มที่ไม่มีอยู่จริงในทางปฏิบัติ

silhouette บอกว่าโครงสร้างทางเรขาคณิตของข้อมูลมี 4 กลุ่ม ซึ่งเป็นข้อเท็จจริง
แต่ 'จำนวนกลุ่มที่ควรใช้' เป็นการตัดสินใจเชิงปฏิบัติการ ไม่ใช่ข้อเท็จจริงทางสถิติ

ข้อควรระวัง: ต้องบันทึกไว้ว่าเรารู้ว่ามี 4 กลุ่มแต่เลือกใช้ 3
ไม่ใช่แกล้งทำเป็นว่าข้อมูลมี 3 กลุ่ม — ความแตกต่างนี้สำคัญตอนทบทวนปีหน้า

คำตอบข้อ 2 — ข้อจำกัดของ K-Means ที่ปรากฏในโจทย์นี้
-----------------------------------------------------
1. ต้องกำหนด k ล่วงหน้า และไม่มีทางบอกได้เองว่า 'ไม่มีกลุ่ม'
2. ไวต่อสเกลของตัวแปรอย่างรุนแรง ดังที่พิสูจน์ในงานที่ 1 และ 2
3. สมมติว่ากลุ่มมีรูปทรงกลมและขนาดใกล้เคียงกัน จึงแบ่งกลุ่มรูปยาวหรือกลุ่มเล็กมากไม่ได้
4. ไม่มีแนวคิดเรื่อง 'ค่าผิดปกติ' — ทุกจุดต้องถูกจับใส่กลุ่มใดกลุ่มหนึ่งเสมอ
   ลูกค้าที่แปลกประหลาดจริง ๆ จึงไปบิดค่าเฉลี่ยของกลุ่มที่มันถูกยัดเข้าไป

คำตอบข้อ 3
----------
Hierarchical : ไม่ต้องกำหนด k ล่วงหน้า และ dendrogram ทำให้เห็นว่ากลุ่มใดใกล้กัน
               จึงตอบคำถามข้อ 1 ได้โดยตรงว่าควรรวมกลุ่มใดเข้าด้วยกัน
DBSCAN       : แยก noise ออกได้ และบอกได้ว่า 'ไม่มีกลุ่ม' ถ้าความหนาแน่นไม่ถึงเกณฑ์
               จึงแก้ข้อจำกัดข้อ 1 และ 4 ได้ แต่ต้องจูน eps ซึ่งไวต่อสเกลไม่แพ้กัน

คำตอบข้อ 4
----------
รันใหม่ทุกไตรมาสเป็นค่าตั้งต้นที่สมเหตุสมผลสำหรับธุรกิจค้าปลีก
แต่สิ่งที่ควรใช้ตัดสินจริงคือสัญญาณเหล่านี้
  · สัดส่วนลูกค้าที่ย้ายกลุ่มเกิน 15% เทียบกับรอบก่อน
  · ค่าเฉลี่ยของกลุ่มใดกลุ่มหนึ่งเลื่อนเกิน 1 ส่วนเบี่ยงเบนมาตรฐาน
  · silhouette ของโมเดลเดิมบนข้อมูลใหม่ลดลงชัดเจน
  · ทีมการตลาดรายงานว่าแคมเปญของกลุ่มหนึ่งได้ผลตอบรับต่างจากเดิมมาก
""")

---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — พิสูจน์ผลของสเกลด้วยตัวเลข | 3 |
| งานที่ 2 — เทียบโปรไฟล์สองแบบและระบุกลุ่มที่หายไป | 4 |
| งานที่ 3 — เส้นโค้ง inertia/silhouette และเหตุผลที่ inertia ใช้ไม่ได้ | 3 |
| งานที่ 4 — แสดงกับดัก silhouette ด้วยตัวเลขและอธิบายได้ | 3 |
| งานที่ 5 — แคมเปญรายกลุ่มพร้อมประมาณการมูลค่า | 4 |
| งานที่ 6 — ข้อจำกัด ทางเลือกอื่น และรอบการทบทวน | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/segment-studio`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง

> 💡 **หมายเหตุเรื่องการเทียบตัวเลขกับสื่อจำลอง**
> สื่อจำลองใช้ K-Means ที่กำหนดค่าเริ่มต้นของ centroid แบบตายตัว ส่วน Lab นี้ใช้
> `KMeans` ของ scikit-learn ที่ใช้ k-means++ และรัน 10 รอบเลือกผลที่ดีที่สุด
>
> **ค่าที่ต้องตรงกัน:** silhouette ของแต่ละ k · ค่า k ที่ดีที่สุด · ขนาดและโปรไฟล์ของแต่ละกลุ่ม
> **ค่าที่อาจต่างในทศนิยมท้าย ๆ:** inertia — เพราะขึ้นกับค่าเริ่มต้น
>
> ความต่างนี้เองเป็นบทเรียน: **K-Means ไวต่อค่าเริ่มต้น** จึงต้องตั้ง `n_init` และ
> `random_state` เสมอ มิฉะนั้นผลจะไม่ซ้ำเดิมแม้รันบนข้อมูลชุดเดียวกัน